# 🚀 SalesTeam AI — Résumé Complet du Projet

> Un système d'intelligence artificielle conçu pour aider les équipes commerciales à recommander les bons produits à chaque client, au bon moment.

---

## 🏗️ Architecture Globale du Projet

```
salesteam_ai/
├── data/
│   ├── raw/             ← Fichiers Excel bruts (ne jamais modifier)
│   └── processed/       ← Fichiers CSV nettoyés (générés automatiquement)
├── src/
│   ├── data/            ← Couche 1 : Chargement & Nettoyage
│   ├── features/        ← Couche 2 : Feature Engineering
│   ├── models/          ← Couche 3 : IA & Machine Learning
│   ├── services/        ← Couche 4 : Logique Métier (recommandations)
│   └── api/             ← Couche 5 : Interface Web (API REST)
├── models/              ← Modèles entraînés sauvegardés (.pkl)
├── notebooks/           ← Notebooks d'exploration et de visualisation
└── requirements.txt     ← Dépendances Python du projet
```

---

## 📂 COUCHE 1 — Données (`src/data/`)

Cette couche est la fondation du projet. Elle lit les fichiers bruts Excel, les nettoie, et les sauvegarde en CSV propres.

---

### 📄 `src/data/loader.py`

**Rôle : Lire et joindre les données brutes**

| Fonction | Ce qu'elle fait |
|---|---|
| `load_factures()` | Lit le fichier Excel des factures/commandes. Renomme les colonnes en français standard. |
| `load_lignes()` | Lit le fichier Excel des lignes de commandes (les articles achetés). |
| `load_clients()` | Lit le fichier Excel des coordonnées GPS des clients. |
| `build_main_table()` | **Clé centrale** : joint les 3 tables sur `(code_facture + societe)`. |

> **Note sur la double clé :** Même si le projet se concentre exclusivement sur la filiale **LSAT**, la jointure sur `(code_facture + societe)` est maintenue par rigueur et robustesse pour éviter tout mélange et faciliter des extensions futures.

---

### 📄 `src/data/cleaner.py`

**Rôle : Nettoyer et standardiser les données**

| Fonction | Ce qu'elle fait |
|---|---|
| `clean_commandes()` | Supprime les factures sans numéro/client. Normalise les espaces. Crée les colonnes temporelles (mois, année...). |
| `clean_lignes()` | Supprime les doublons (934 trouvés). Récupère les catégories manquantes via le mode historique (165 récupérées). Flag les commandes en gros volume (`is_bulk_order`). |
| `clean_gps()` | Nettoie les coordonnées GPS. |
| `clean_all()` | Orchestre les 3 fonctions ci-dessus et sauvegarde en CSV dans `data/processed/`. |

**Commande d'exécution :**
```bash
python -m src.data.cleaner
```

## 📂 COUCHE 2 — Feature Engineering (`src/features/`)

Cette couche transforme les transactions individuelles en une **matrice d'apprentissage** : 1 ligne = 1 paire (client, article) avec toutes ses caractéristiques calculées.

---

### 📄 `src/features/feature_engineering.py`

**Rôle : Créer les 29 variables (Features) pour le modèle ML**

| Groupe | Variables calculées |
|---|---|
| **Groupe 1 — Historique** | `avg_qty`, `std_qty`, `min_qty`, `max_qty`, `total_qty`, `frequency`, `last_qty`, `recency_days`, `avg_delay_days`, `trend` |
| **Groupe 2 — Saisonnalité** | `current_month_coef`, `avg_seasonal_coef`, `best_month` |
| **Groupe 3 — Géographie** | `has_gps`, `latitude`, `longitude` |
| **Groupe 4 — Profil Produit** | `categorie`, `designation`, `is_bulk_product`, `nb_clients`, `days_since_first_order`, `is_new_product` |
| **Groupe 5 — Profil Client** | `client_total_products`, `client_total_invoices`, `client_avg_basket_size` |

**Résultat :** `data/processed/feature_matrix.csv` — **40 765 lignes × 29 colonnes**

**Commande d'exécution :**
```bash
python -m src.features.feature_engineering
```

---

### 📄 `src/features/seasonality.py`

**Rôle : Calculer dynamiquement les coefficients de saisonnalité (inclus le Ramadan)**

| Fonction | Ce qu'elle fait |
|---|---|
| `compute_seasonal_coefficients()` | Analyse l'historique réel des ventes pour apprendre les mois forts/faibles. Ex: Juillet = +25%. |
| `get_ramadan_months()` | Convertit les dates en calendrier islamique pour détecter automatiquement les mois du Ramadan. |
| `apply_ramadan_boost()` | Applique +15% sur les coefficients des mois concernés par le Ramadan. |
| `get_month_coefficients()` | Façade principale : appelle les 3 fonctions ci-dessus en une seule étape. |

## 📂 COUCHE 3 — Modèles IA (`src/models/`)

Cette couche contient les algorithmes de Machine Learning qui apprennent les comportements d'achat et font les prédictions.

---

### 📄 `src/models/train_classifier.py`

**Rôle : Entraîner un classifieur binaire XGBoost**  
Pour chaque paire (client, article), prédit : **va-t-il commander ce produit dans les 30 prochains jours ?** (`1` = Oui / `0` = Non)

| Fonction | Ce qu'elle fait |
|---|---|
| `prepare_classifier_dataset()` | Crée la variable cible `will_order` sans fuite de données temporelle. |
| `train_classifier()` | Entraîne le modèle XGBoost avec correction du déséquilibre de classes. |
| `evaluate_classifier()` | Calcule Accuracy, Precision, Recall, F1, ROC-AUC et la matrice de confusion. |
| `load_classifier()` | Charge un modèle sauvegardé depuis `models/classifier_lsat.pkl`. |

> **Statut actuel :** Fonctions vides (`NotImplementedError`) — prêtes à être implémentées.

---

### 📄 `src/models/train_regressor.py`

**Rôle : Prédire la quantité à commander**  
Une fois qu'on sait qu'un client va commander un produit, ce modèle prédit **combien d'unités** il va commander.

> **Statut actuel :** Squelette vide — à implémenter.

---

### 📄 `src/models/train_svd.py`

**Rôle : Filtrage collaboratif (Collaborative Filtering)**  
Un modèle SVD (comme Netflix) qui recommande des produits en se basant sur la similitude de comportement entre clients ("les clients qui ressemblent à vous ont aussi acheté...").

> **Statut actuel :** Squelette vide — à implémenter.

---

### 📄 `src/models/predictor.py`

**Rôle : Point d'entrée unique pour toutes les prédictions**  
Orchestre les 3 modèles (Classifieur + Régresseur + SVD) et applique la logique de cold start pour les nouveaux clients/produits.

| Méthode | Ce qu'elle fait |
|---|---|
| `__init__()` | Charge tous les modèles en mémoire au démarrage de l'API. |
| `predict(client_id)` | Génère la liste des recommandations classées pour un client donné. |
| `_blend_scores()` | Combine le score du classifieur (60%) et du SVD (40%) en un seul score de confiance. |

> **Statut actuel :** Squelette vide — à implémenter.

---

### 📄 `src/models/cold_start.py`

**Rôle : Gérer les nouveaux clients ou nouveaux produits sans historique**  
Quand un nouveau client n'a aucun historique d'achat, le modèle principal ne peut pas prédire. Ce module applique des règles de fallback basées sur la popularité, la région géographique, etc.

> **Statut actuel :** Squelette vide — à implémenter.

## 📂 COUCHE 4 — Services Métier (`src/services/`)

Cette couche contient la logique de présentation des résultats de l'IA à l'utilisateur final (le commercial).

---

### 📄 `src/services/recommendation.py`
**Rôle :** Formater et filtrer la liste des recommandations brutes du modèle en un résultat lisible par l'API.

### 📄 `src/services/explanation.py`
**Rôle :** Générer des explications en langage naturel des recommandations. Ex: *"Ce produit est recommandé car ce client l'a acheté 8 fois et sa dernière commande date d'il y a 3 jours."*

### 📄 `src/services/feedback.py`
**Rôle :** Enregistrer les retours des commerciaux ("Ce produit a été acheté", "Ce produit a été refusé") pour améliorer le modèle en continu (boucle d'apprentissage).

## 📂 COUCHE 5 — API Web (`src/api/`)

L'interface qui expose le système d'IA aux applications externes (application mobile des commerciaux, ERP, etc.) via des requêtes HTTP.

---

### 📄 `src/api/main.py`
**Rôle :** Point d'entrée FastAPI. Démarre le serveur web et charge les modèles en mémoire au démarrage.

### 📄 `src/api/schemas.py`
**Rôle :** Définit la structure exacte des requêtes et des réponses de l'API (validation Pydantic). Ex: le format exact du JSON retourné par `/recommend`.

### 📄 `src/api/routes/`
**Rôle :** Contient les endpoints HTTP (`/recommend`, `/feedback`, etc.).

**Commande pour démarrer le serveur :**
```bash
uvicorn src.api.main:app --reload
```

## 📂 Notebooks (`notebooks/`)

| Notebook | Rôle |
|---|---|
| `03_processed_data_visualization.ipynb` | Visualisation graphique des 4 tables traitées (courbes, camemberts, heatmap de corrélation, carte GPS). |
| `04_processed_data_preview.ipynb` | Aperçu rapide en tableau des 5 premières lignes de chaque table traitée. |

---

## 🗺️ Flux de Données Complet

```
[Fichiers Excel bruts]
        ↓  loader.py (lit + joint)
[Tables Python en mémoire]
        ↓  cleaner.py (nettoie + valide)
[CSV Propres dans data/processed/]
        ↓  feature_engineering.py (agrège + calcule)
[feature_matrix.csv — 40 765 × 29]
        ↓  train_classifier.py (apprend)
[Modèle Entraîné : models/classifier_lsat.pkl]
        ↓  predictor.py (prédit)
[Liste de recommandations classées par score]
        ↓  services/recommendation.py (formate)
        ↓  services/explanation.py (explique)
[API FastAPI /recommend]
        ↓
[Application Mobile du Commercial]
```

---

## ✅ État d'Avancement du Projet

| Composant | Statut |
|---|---|
| `loader.py` | ✅ Complet |
| `cleaner.py` | ✅ Complet |
| `feature_engineering.py` | ✅ Complet |
| `seasonality.py` | ✅ Complet |
| `train_classifier.py` | 🔲 À implémenter |
| `train_regressor.py` | 🔲 À implémenter |
| `train_svd.py` | 🔲 À implémenter |
| `predictor.py` | 🔲 À implémenter |
| `cold_start.py` | 🔲 À implémenter |
| `services/` | 🔲 À implémenter |
| `api/` | 🔲 À implémenter |